In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Multi-Parameter Learning: Linear Regression from Scratch\n",
    "\n",
    "**Goal:** Train a model to learn the function `y = 2x + 3` using gradient descent.\n",
    "\n",
    "**Key Concepts:**\n",
    "- Learning multiple parameters simultaneously (weight and bias)\n",
    "- Understanding underdetermined vs well-determined systems\n",
    "- Importance of sufficient training data\n",
    "\n",
    "---"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch\n",
    "import matplotlib.pyplot as plt\n",
    "import time"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Part 1: The Problem - Single Data Point (Underdetermined)\n",
    "\n",
    "**What happens when we only have ONE data point?**\n",
    "\n",
    "With one equation and two unknowns, there are **infinite solutions!**"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# True function: y = 2x + 3\n",
    "def true_function(x):\n",
    "    return 2 * x + 3\n",
    "\n",
    "# Single data point\n",
    "x = torch.tensor(4.0)\n",
    "y_true = true_function(x)  # = 11.0\n",
    "\n",
    "print(f\"Input: {x}\")\n",
    "print(f\"True output: {y_true}\")\n",
    "print(f\"\\nGoal: Learn that y = 2*x + 3\")\n",
    "print(f\"Challenge: We only have ONE data point!\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Model parameters (start with wrong guesses)\n",
    "weight = torch.tensor(0.5, requires_grad=True)\n",
    "bias = torch.tensor(0.0, requires_grad=True)\n",
    "\n",
    "print(f\"Starting weight: {weight.item()}\")\n",
    "print(f\"Starting bias: {bias.item()}\")\n",
    "\n",
    "learning_rate = 0.01\n",
    "num_steps = 100\n",
    "\n",
    "# Track progress\n",
    "weight_history = []\n",
    "bias_history = []\n",
    "loss_history = []\n",
    "\n",
    "for step in range(num_steps):\n",
    "    # Forward pass: calculate prediction\n",
    "    y_pred = weight * x + bias\n",
    "    \n",
    "    # Calculate loss\n",
    "    loss = (y_pred - y_true) ** 2\n",
    "    \n",
    "    # Backward pass: compute gradients\n",
    "    loss.backward()\n",
    "    \n",
    "    # Update parameters\n",
    "    with torch.no_grad():\n",
    "        weight = weight - learning_rate * weight.grad\n",
    "        weight.requires_grad = True\n",
    "        \n",
    "        bias = bias - learning_rate * bias.grad\n",
    "        bias.requires_grad = True\n",
    "    \n",
    "    # Clear gradients\n",
    "    weight.grad = None\n",
    "    bias.grad = None\n",
    "    \n",
    "    # Track progress\n",
    "    weight_history.append(weight.item())\n",
    "    bias_history.append(bias.item())\n",
    "    loss_history.append(loss.item())\n",
    "    \n",
    "    if (step + 1) % 20 == 0:\n",
    "        print(f\"Step {step+1}: weight={weight.item():.4f}, bias={bias.item():.4f}, loss={loss.item():.4f}\")\n",
    "\n",
    "print(f\"\\nFinal weight: {weight.item():.4f} (should be 2.0)\")\n",
    "print(f\"Final bias: {bias.item():.4f} (should be 3.0)\")\n",
    "print(f\"\\n⚠️ Loss is 0, but values are WRONG! Why?\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Why Did This Happen?\n",
    "\n",
    "**The model found A valid solution, not THE solution:**\n",
    "\n",
    "```\n",
    "With one data point: weight × 4 + bias = 11\n",
    "\n",
    "Infinite solutions:\n",
    "- weight=2.0, bias=3.0 → 2×4+3 = 11 ✅\n",
    "- weight=2.6, bias=0.5 → 2.6×4+0.5 ≈ 11 ✅\n",
    "- weight=1.0, bias=7.0 → 1×4+7 = 11 ✅\n",
    "```\n",
    "\n",
    "**This is called an UNDERDETERMINED SYSTEM:**\n",
    "- More unknowns (2) than equations (1)\n",
    "- Model finds the easiest path from starting point\n",
    "\n",
    "---"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Part 2: The Solution - Multiple Data Points\n",
    "\n",
    "**Fix: Give the model MORE data!**\n",
    "\n",
    "With multiple points, there's only ONE solution that fits them all."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Multiple data points (6 examples)\n",
    "X = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])\n",
    "Y = 2 * X + 3  # True function: [5, 7, 9, 11, 13, 15]\n",
    "\n",
    "print(\"Training data:\")\n",
    "for i in range(len(X)):\n",
    "    print(f\"  x={X[i].item()}, y={Y[i].item()}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Reset parameters\n",
    "weight = torch.tensor(0.5, requires_grad=True)\n",
    "bias = torch.tensor(0.0, requires_grad=True)\n",
    "\n",
    "learning_rate = 0.01\n",
    "num_steps = 200\n",
    "\n",
    "weight_history = []\n",
    "bias_history = []\n",
    "loss_history = []\n",
    "\n",
    "for step in range(num_steps):\n",
    "    # Forward pass: predictions for ALL data points\n",
    "    y_pred = weight * X + bias\n",
    "    \n",
    "    # Loss: average across all points\n",
    "    loss = ((y_pred - Y) ** 2).mean()\n",
    "    \n",
    "    # Backward pass\n",
    "    loss.backward()\n",
    "    \n",
    "    # Update parameters\n",
    "    with torch.no_grad():\n",
    "        weight = weight - learning_rate * weight.grad\n",
    "        weight.requires_grad = True\n",
    "        \n",
    "        bias = bias - learning_rate * bias.grad\n",
    "        bias.requires_grad = True\n",
    "    \n",
    "    # Clear gradients\n",
    "    weight.grad = None\n",
    "    bias.grad = None\n",
    "    \n",
    "    # Track progress\n",
    "    weight_history.append(weight.item())\n",
    "    bias_history.append(bias.item())\n",
    "    loss_history.append(loss.item())\n",
    "    \n",
    "    if (step + 1) % 50 == 0:\n",
    "        print(f\"Step {step+1}: weight={weight.item():.4f}, bias={bias.item():.4f}, loss={loss.item():.6f}\")\n",
    "\n",
    "print(f\"\\nFinal weight: {weight.item():.4f} (target: 2.0)\")\n",
    "print(f\"Final bias: {bias.item():.4f} (target: 3.0)\")\n",
    "print(f\"\\n✅ Much closer to the true values!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Part 3: Visualize the Learning Process"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Plot learning curves\n",
    "fig, axes = plt.subplots(1, 3, figsize=(15, 4))\n",
    "\n",
    "# Weight convergence\n",
    "axes[0].plot(weight_history, linewidth=2)\n",
    "axes[0].axhline(y=2.0, color='r', linestyle='--', label='Target (2.0)')\n",
    "axes[0].set_xlabel('Step')\n",
    "axes[0].set_ylabel('Weight Value')\n",
    "axes[0].set_title('Weight Learning Progress')\n",
    "axes[0].legend()\n",
    "axes[0].grid(True, alpha=0.3)\n",
    "\n",
    "# Bias convergence\n",
    "axes[1].plot(bias_history, color='orange', linewidth=2)\n",
    "axes[1].axhline(y=3.0, color='r', linestyle='--', label='Target (3.0)')\n",
    "axes[1].set_xlabel('Step')\n",
    "axes[1].set_ylabel('Bias Value')\n",
    "axes[1].set_title('Bias Learning Progress')\n",
    "axes[1].legend()\n",
    "axes[1].grid(True, alpha=0.3)\n",
    "\n",
    "# Loss curve\n",
    "axes[2].plot(loss_history, color='green', linewidth=2)\n",
    "axes[2].set_xlabel('Step')\n",
    "axes[2].set_ylabel('Loss')\n",
    "axes[2].set_title('Loss Over Time')\n",
    "axes[2].set_yscale('log')\n",
    "axes[2].grid(True, alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../results/charts/multi_parameter_learning.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "print(\"\\n📊 Chart saved: results/charts/multi_parameter_learning.png\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Part 4: Visualize the Fitted Line"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Plot data and learned function\n",
    "plt.figure(figsize=(10, 6))\n",
    "\n",
    "# Training data points\n",
    "plt.scatter(X.numpy(), Y.numpy(), s=100, color='blue', label='Training Data', zorder=3)\n",
    "\n",
    "# True function\n",
    "x_range = torch.linspace(0, 7, 100)\n",
    "y_true_line = 2 * x_range + 3\n",
    "plt.plot(x_range.numpy(), y_true_line.numpy(), 'r--', linewidth=2, label='True: y=2x+3', alpha=0.7)\n",
    "\n",
    "# Learned function\n",
    "y_learned = weight.item() * x_range + bias.item()\n",
    "plt.plot(x_range.numpy(), y_learned.numpy(), 'g-', linewidth=2, \n",
    "         label=f'Learned: y={weight.item():.2f}x+{bias.item():.2f}')\n",
    "\n",
    "plt.xlabel('x', fontsize=12)\n",
    "plt.ylabel('y', fontsize=12)\n",
    "plt.title('Linear Regression: Learned vs True Function', fontsize=14, fontweight='bold')\n",
    "plt.legend(fontsize=11)\n",
    "plt.grid(True, alpha=0.3)\n",
    "plt.savefig('../results/charts/linear_regression_fit.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "print(\"\\n📊 Chart saved: results/charts/linear_regression_fit.png\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Key Takeaways\n",
    "\n",
    "### 1️⃣ Data Requirements\n",
    "**With 1 data point:**\n",
    "- Infinite valid solutions\n",
    "- Model finds ANY solution that works\n",
    "- Loss = 0, but parameters are wrong!\n",
    "\n",
    "**With multiple data points:**\n",
    "- Usually one unique solution\n",
    "- Model must find the true pattern\n",
    "- Parameters converge to correct values\n",
    "\n",
    "### 2️⃣ The Learning Pattern\n",
    "```python\n",
    "for step in range(num_steps):\n",
    "    # 1. Forward pass\n",
    "    prediction = model(input)\n",
    "    \n",
    "    # 2. Calculate loss\n",
    "    loss = (prediction - target) ** 2\n",
    "    \n",
    "    # 3. Backward pass\n",
    "    loss.backward()\n",
    "    \n",
    "    # 4. Update parameters\n",
    "    param = param - lr * param.grad\n",
    "    \n",
    "    # 5. Clear gradients\n",
    "    param.grad = None\n",
    "```\n",
    "\n",
    "**This same pattern powers ALL neural networks!**\n",
    "\n",
    "### 3️⃣ Why This Matters\n",
    "- Real neural networks have millions of parameters\n",
    "- Need LOTS of data to constrain the solution space\n",
    "- This is why \"big data\" matters in ML!\n",
    "\n",
    "### 4️⃣ Next Steps\n",
    "- Scale to more complex functions (non-linear)\n",
    "- Add more parameters (neural network layers)\n",
    "- Train on real datasets (images, text)\n",
    "\n",
    "---\n",
    "\n",
    "**You just built a working machine learning algorithm from scratch! 🎉**"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.11.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}